# nDCG@5 QA Validation

## 1. Purpose

nDCG@5 measures ranking quality, not merely the number of relevant documents. It rewards rankings that place relevant documents earlier in the returned list.


## 2. Imports and output location

The helper locates the repository root whether Jupyter starts at the repository root or inside `notebooks/qa`.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from idp_eval import EvaluationCase, EvaluationFramework, create_azure_judge
from idp_eval.judges import AzureJudgeConfig

from idp_eval import NDCGAtKEvaluator


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "idp_eval").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the idp-eval repository.")


REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / "qa_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Configure the judge

Replace every placeholder before running. No Phoenix server is required. If your
application already constructs a compatible judge, you may replace this cell
with that existing construction. Keep credentials in your application's secret
management system rather than saving them in this notebook.


In [ ]:
azure_config = AzureJudgeConfig(
    model="YOUR_AZURE_DEPLOYMENT",
    azure_endpoint="YOUR_AZURE_ENDPOINT",
    tenant_id="YOUR_TENANT_ID",
    client_id="YOUR_CLIENT_ID",
    client_secret="YOUR_CLIENT_SECRET",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)

judge = create_azure_judge(config=azure_config)


## 4. Five mock evaluation cases

These cases intentionally span clear pass, partial, and fail behaviors. `expected_behavior` is a human QA aid, not an exact model-score assertion.


In [ ]:
cases = [
    EvaluationCase(
        case_id="NDCG-001",
        input="How do customers pay an invoice?",
        retrieved_documents=[
            {"document_id": "inv-1", "text": "Pay invoices by card or ACH in the billing portal.", "score": 0.99},
            {"document_id": "inv-2", "text": "The portal displays invoice payment confirmation.", "score": 0.95},
            {"document_id": "inv-3", "text": "Use the invoice number for bank payments.", "score": 0.91},
            {"document_id": "vac-1", "text": "Employees receive 20 vacation days.", "score": 0.87},
            {"document_id": "db-1", "text": "Database backups run nightly.", "score": 0.83},
        ],
    ),
    EvaluationCase(
        case_id="NDCG-002",
        input="How do customers pay an invoice?",
        retrieved_documents=[
            {"document_id": "inv-4", "text": "Pay invoices by card or ACH in the billing portal.", "score": 0.99},
            {"document_id": "ship-1", "text": "Update a shipping address before dispatch.", "score": 0.95},
            {"document_id": "db-2", "text": "Database backups run nightly.", "score": 0.91},
            {"document_id": "inv-5", "text": "The portal displays invoice payment confirmation.", "score": 0.87},
            {"document_id": "inv-6", "text": "Use the invoice number for bank payments.", "score": 0.83},
        ],
    ),
    EvaluationCase(
        case_id="NDCG-003",
        input="How do customers pay an invoice?",
        retrieved_documents=[
            {"document_id": "vac-2", "text": "Employees receive 20 vacation days.", "score": 0.99},
            {"document_id": "db-3", "text": "Database backups run nightly.", "score": 0.95},
            {"document_id": "inv-7", "text": "Pay invoices by card or ACH in the billing portal.", "score": 0.91},
            {"document_id": "inv-8", "text": "The portal displays invoice payment confirmation.", "score": 0.87},
            {"document_id": "inv-9", "text": "Use the invoice number for bank payments.", "score": 0.83},
        ],
    ),
    EvaluationCase(
        case_id="NDCG-004",
        input="How do customers pay an invoice?",
        retrieved_documents=[
            {"document_id": "inv-10", "text": "Pay invoices by card or ACH in the billing portal.", "score": 0.99},
            {"document_id": "pw-1", "text": "Reset a forgotten password from the sign-in page.", "score": 0.95},
            {"document_id": "inv-11", "text": "Use the invoice number for bank payments.", "score": 0.91},
            {"document_id": "vac-3", "text": "Employees receive 20 vacation days.", "score": 0.87},
            {"document_id": "inv-12", "text": "The portal displays invoice payment confirmation.", "score": 0.83},
        ],
    ),
    EvaluationCase(
        case_id="NDCG-005",
        input="How do customers pay an invoice?",
        retrieved_documents=[
            {"document_id": "pw-2", "text": "Reset a forgotten password from the sign-in page.", "score": 0.99},
            {"document_id": "vac-4", "text": "Employees receive 20 vacation days.", "score": 0.95},
            {"document_id": "db-4", "text": "Database backups run nightly.", "score": 0.91},
            {"document_id": "ship-2", "text": "Update a shipping address before dispatch.", "score": 0.87},
            {"document_id": "sec-1", "text": "Administrators must use MFA.", "score": 0.83},
        ],
    ),
]

expected_behavior = {
    "NDCG-001": "ideal/high — all relevant documents ranked first",
    "NDCG-002": "slightly degraded — relevant documents misplaced lower",
    "NDCG-003": "poor ranking — relevant documents mostly near the bottom",
    "NDCG-004": "mixed/suboptimal ranking",
    "NDCG-005": "no_relevant_retrieved",
}


## 5. Inspect the mock inputs


In [ ]:
case_rows = []
for case in cases:
    case_rows.append({
        "case_id": case.case_id,
        "expected_behavior": expected_behavior[case.case_id],
        "input": getattr(case, "input"),
        "retrieved_documents": getattr(case, "retrieved_documents"),
    })

cases_df = pd.DataFrame(case_rows)
display(cases_df)


## 6. Configure one evaluator and Excel output

This notebook runs exactly one metric. `resume=False` creates a fresh QA workbook and no Phoenix tracing is configured.


In [ ]:
excel_path = OUTPUT_DIR / "ndcg_at_k_validation.xlsx"
evaluator = NDCGAtKEvaluator(k=5, verbose=True)
framework = EvaluationFramework(
    evaluators=[evaluator],
    judge=judge,
    output="excel",
    excel_path=str(excel_path),
    resume=False,
)

results = framework.evaluate_many(
    cases,
    run_name="qa-validation",
    dataset_name="mock-acceptance-cases",
    show_progress=True,
)


## 7. Result summary


In [ ]:
METRIC_NAME = "ndcg_at_5"
summary_rows = []
for case, result_map in zip(cases, results, strict=True):
    result = result_map[METRIC_NAME]
    summary_rows.append({
        "case_id": case.case_id,
        "expected_behavior": expected_behavior[case.case_id],
        "score": result.score,
        "label": result.label,
        "explanation": result.explanation,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


## 8. Inspect Excel output

The workbook summary is in `evaluations`; item-level evidence is in `retrieval_documents`.


In [ ]:
evaluations_df = pd.read_excel(excel_path, sheet_name="evaluations")
display(evaluations_df)

details_df = pd.read_excel(excel_path, sheet_name="retrieval_documents")

visible_columns = [
    column
    for column in (
        "key_id",
        "rank",
        "document_id",
        "text",
        "relevant",
        "relevance_score",
        "reason",
        "retrieval_score",
    )
    if column in details_df.columns
]
display(details_df[visible_columns])


## 9. Sanity assertions

These assertions validate framework/output behavior and broad direction only; they do not require exact LLM-generated fractions.


In [ ]:
assert len(results) == 5
assert excel_path.exists()
assert all(METRIC_NAME in result_map for result_map in results)
assert len(evaluations_df) == 5
assert set(evaluations_df["key_id"]) == {case.case_id for case in cases}
assert set(("text", "relevant", "relevance_score", "reason")).issubset(details_df.columns)
assert results[0][METRIC_NAME].score >= results[2][METRIC_NAME].score
assert results[4][METRIC_NAME].label == "no_relevant_retrieved"
print("nDCG@5 QA sanity checks passed.")


## 10. Close judge resources and report the workbook path


In [ ]:
judge.close()
print(f"Excel output: {excel_path.resolve()}")
